# 01 — Retailrocket: исследование данных

## Цель и постановка задачи

Исследуем журнал неявной обратной связи интернет-магазина и формулируем задачу как рекомендацию **Top-10 новых для пользователя товаров**. `view`, `addtocart` и `transaction` — события разной силы намерения; далее они кодируются весами 1, 3 и 5. Повторные события одного пользователя и товара суммируются при построении профиля.

В этом ноутбуке сначала изучаются полные данные, а затем проверяется подготовленный хронологический split. Модели не обучаются на validation/test. Визуализации сохраняются в `outputs/figures/` и одновременно отображаются здесь.

## Данные и временная схема

Источник — публичный Retailrocket events.csv. `timestamp` — Unix epoch в миллисекундах; загрузчик из `src/retailrocket.py` проверяет схему и пропуски и переводит время в UTC. Разбиение построено по **уникальным timestamp** в порядке времени (70/15/15), поэтому события с одной временной меткой не могут попасть в разные части.

Запустите сначала `python -m scripts.run_eda`, если файлов в `data/processed/` ещё нет. Это ноутбук-исследование; сложная логика загрузки и split остаётся в модуле `src/`.

In [ ]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
plt.rcParams.update({"figure.dpi": 300, "savefig.dpi": 300})
from IPython.display import display

ROOT = Path.cwd()
if not (ROOT / "src").is_dir():
    ROOT = ROOT.parent
DATA_PATH = ROOT / "data" / "raw" / "events.csv"
FIGURES = ROOT / "outputs" / "figures"
FIGURES.mkdir(parents=True, exist_ok=True)

from src.retailrocket import load_events, load_processed_events

events = load_events(DATA_PATH)
print("Loaded {:,} events from {}".format(len(events), DATA_PATH))

## Обзор и качество наблюдений

Считаем уникальных посетителей, товаров и пар user-item. Разреженность показывает, насколько малую часть декартова произведения «посетитель × товар» мы наблюдаем; она объясняет выбор методов, работающих со sparse-матрицами, а не с плотной таблицей.

In [ ]:
event_counts = events["event"].value_counts().sort_values(ascending=False)
user_activity = events.groupby("visitorid").size()
item_activity = events.groupby("itemid").size()
unique_pairs = events[["visitorid", "itemid"]].drop_duplicates().shape[0]
n_users = int(events["visitorid"].nunique())
n_items = int(events["itemid"].nunique())
density = unique_pairs / float(n_users * n_items)

summary = pd.DataFrame({
    "value": [len(events), n_users, n_items, unique_pairs, density,
              events["event_time"].min(), events["event_time"].max()],
}, index=["events", "unique visitors", "unique items", "unique user-item pairs",
         "observed matrix density", "first event (UTC)", "last event (UTC)"])
display(summary)
display(event_counts.rename("events").to_frame())
print("Activity per visitor — median {:.1f}, p90 {:.1f}, max {:,}".format(
    user_activity.median(), user_activity.quantile(0.90), user_activity.max()))
print("Activity per item — median {:.1f}, p90 {:.1f}, max {:,}".format(
    item_activity.median(), item_activity.quantile(0.90), item_activity.max()))

## Типы событий и временная активность

Логарифмическая шкала для числа событий помогает не скрыть редкие, но более сильные сигналы `addtocart` и `transaction` рядом с многочисленными просмотрами. Дневной график помогает увидеть изменения активности и сезонные/календарные эффекты; ось времени приведена к UTC.

In [ ]:
daily = (events.assign(day=events["event_time"].dt.floor("D"))
         .groupby(["day", "event"]).size().unstack(fill_value=0).sort_index())
fig, axes = plt.subplots(2, 1, figsize=(12, 9))
event_counts.sort_values().plot(kind="barh", ax=axes[0], color="#4472C4")
axes[0].set_xscale("log")
axes[0].set(title="События по типам (логарифмическая шкала)", xlabel="Количество событий", ylabel="")
daily.plot(ax=axes[1], linewidth=1.2)
axes[1].set(title="Активность по дням (UTC)", xlabel="Дата", ylabel="События")
axes[1].legend(title="Тип события")
fig.tight_layout()
fig.savefig(FIGURES / "notebook_eda_events_time.png", dpi=300, bbox_inches="tight")
plt.show()
plt.close(fig)

## Распределения активности и разреженность

Распределения имеют длинный хвост: большинство пользователей и товаров встречаются редко, а небольшая доля — очень часто. Поэтому отдельно рассматриваем частоты в логарифмических координатах и концентрацию взаимодействий у популярных товаров. Эти графики описывают только данные и не используются для настройки модели.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
for values, axis, title, xlabel in [
    (user_activity, axes[0], "События на пользователя", "События"),
    (item_activity, axes[1], "События на товар", "События"),
]:
    bins = np.geomspace(1, max(int(values.max()) + 1, 2), 35)
    axis.hist(values, bins=bins, color="#ED7D31", edgecolor="white")
    axis.set_xscale("log")
    axis.set_yscale("log")
    axis.set(title=title, xlabel=xlabel, ylabel="Частота (лог.)")
fig.tight_layout()
fig.savefig(FIGURES / "notebook_eda_activity_distributions.png", dpi=300, bbox_inches="tight")
plt.show()
plt.close(fig)

item_counts = item_activity.sort_values(ascending=False)
cumulative_share = item_counts.cumsum() / item_counts.sum()
fig, axis = plt.subplots(figsize=(9, 5))
axis.plot(np.arange(1, len(cumulative_share) + 1), cumulative_share.values, color="#70AD47")
axis.set(xscale="log", ylim=(0, 1.02), title="Концентрация событий по популярности товаров",
         xlabel="Число самых популярных товаров (лог.)", ylabel="Доля всех событий")
axis.grid(alpha=0.25)
fig.tight_layout()
fig.savefig(FIGURES / "notebook_eda_item_concentration.png", dpi=300, bbox_inches="tight")
plt.show()
plt.close(fig)
print("Уникальных пар user-item: {:,}; плотность user-item матрицы: {:.8%}.".format(
    unique_pairs, density))

## Проверка хронологического split и выводы EDA

Временные границы — часть протокола оценки: validation используется при выборе модели, test остаётся нетронутым до финальной оценки. Дополнительно проверяем реальные крайние timestamp сохранённых частей, а не только доверяем именам файлов.

In [ ]:
processed = ROOT / "data" / "processed"
train = load_processed_events(processed / "train.csv")
validation = load_processed_events(processed / "validation.csv")
test = load_processed_events(processed / "test.csv")
manifest = json.loads((processed / "split_manifest.json").read_text())

bounds = pd.DataFrame({
    "rows": [len(train), len(validation), len(test)],
    "min UTC": [train.event_time.min(), validation.event_time.min(), test.event_time.min()],
    "max UTC": [train.event_time.max(), validation.event_time.max(), test.event_time.max()],
}, index=["train", "validation", "test"])
display(bounds)
assert train.event_time.max() < validation.event_time.min()
assert validation.event_time.max() < test.event_time.min()
assert len(train) + len(validation) + len(test) == len(events)
print("Проверки прошли: временные интервалы строго разделены; все строки учтены.")

findings = [
    "Сигналы сильно несбалансированы: просмотры доминируют над корзинами и покупками.",
    "Данные user-item крайне разрежены, поэтому применяем sparse collaborative filtering.",
    "Активность пользователей и товаров имеет длинный хвост; далее ограничиваем offline benchmark фиксированной популярной частью каталога и воспроизводимой когортой.",
    "Оценка отделяет train, validation и test по времени; test нельзя использовать для выбора параметров.",
]
for finding in findings:
    print("• " + finding)